# Solar filament segmentation: two-model Mask2Former ensemble with a learned candidate scorer

This notebook rebuilds the final submission **from the raw photos to the CSV**. That covers the training of every model, the inference, the
learned candidate scorer, the selection, the boundary redraw, the checks and the illustrations.

| Submission | Out-of-fold PQ (707 photos) | Kaggle submission ref |
|---|---|---|
| **Two-model ensemble, learned scorer, 3 px boundary redraw** | **0.4770** | 58304440 |

Every model is trained on the 5 folds of `data/folds.json` (grouped by date). The models, epochs, scorer settings and selection
threshold are chosen on **out-of-fold PQ**. For the boundary redraw, 1 px and 3 px are equivalent out of fold (0.4774 vs 0.4770,
difference -0.0003, 95% interval -0.0012 to +0.0006); the final uses 3 px.

## How the pipeline works

```
 photos ──► [1a] Mask2Former (grayscale input) x5 folds ─────► 8 views (D4) ──┐
        ──► [1b] Mask2Former (limb-corrected input) x5 folds ► 8 views (D4) ──┴─► [2c] fused 16-view candidate pool per photo
        ──► [1c] Mask R-CNN x5 + U-Nets cnx / cnxL / r34 x5 ──► detector candidates, probability + spine maps ─┐
        ──► [1d] ROI detector + refiner x5 ──────────────────► ROI candidates ──────────────────────────────────┤
                                                                                                                 ▼
   [3] 76 numbers per candidate (shape, scores, agreement with the other models) ──► LightGBM scorer q (expected matched IoU)
   [4] exact non-overlapping selection (q > tau, overlap < theta) ──► 5 fold models fused on test
   [5] boundary of each kept mask redrawn within 3 px from the U-Net maps ──► outputs/submission.csv
```

* **Out of fold.** Model *k* never sees fold *k*. Every training photo gets predictions from a model that did not train on it, and the
  scorer is trained on those predictions, the same way it is used on test.
* **Test.** Each of the 5 fold models (one per family) proposes candidates. The scorer trained on all out-of-fold rows scores them, and
  the 5 selections are fused into one.

## How to run it

1. Python 3.12 environment: `pip install --extra-index-url https://download.pytorch.org/whl/cu128 -r env/requirements-venv.txt
   -r env/requirements-extra.txt jupyterlab`. Linux, NVIDIA GPU(s) with 12 GB or more, driver with CUDA 12.8.
2. Kaggle API token in `~/.kaggle/kaggle.json`, with the competition rules accepted on kaggle.com.
3. The Google Drive link (or id) of `filament_models.tar` in `reference/DOWNLOADS.json`, or the file itself in `downloads/`.
4. Run all cells (or `python run_headless.py`). Every step skips work whose output exists:
   * section 0 downloads what is missing and checks every file's SHA-256;
   * a model is trained only if its checkpoint is missing or changed, and only after you confirm;
   * the scorer, selection and redraw (steps 3 to 5, ~15 min of CPU) always run, and step 6 compares `outputs/submission.csv` with the
     submitted file byte for byte.

| Step | What | Hardware | Time (4 x RTX 3080 Ti 12 GB, 24 cores) |
|---|---|---|---|
| 1a / 1b | Mask2Former, 2 families x 5 folds | GPU | ~5.5 h per fold per GPU (only if missing) |
| 1c | Mask R-CNN + 3 U-Nets, 5 folds | GPU | ~4.5 h on 4 GPUs (only if missing) |
| 1d | ROI detector + refiner, 5 folds | GPU | ~5 h on 3 GPUs (only if missing) |
| 2 | Inference (candidates, banks, U-Net maps) | GPU | ~25 min with the shipped outputs (U-Net maps only); ~1.5 h from scratch |
| 3-5 | Features, scorer, selection, test fusion, redraw | CPU | ~15 min |
| 6-7 | Scores, checks, illustrations | CPU | ~3 min |

## Repository layout

| Folder | Content |
|---|---|
| `src/` | All the code: `models/` (Mask2Former, Mask R-CNN, U-Net, ROI detector + refiner), `training/`, `inference/`, `scorer/` (features, LightGBM, selection, redraw), `evaluation/` (official metric), `data/`, `utils/`, and `paths.py` (every folder below) |
| `configs/` | Model and data settings |
| `data/` | `folds.json`; the photos and annotations are downloaded from Kaggle |
| `models/` | One folder per trained model, `{model_name}-f{fold}` (downloaded or trained; git-ignored) |
| `work/` | Inference outputs and scorer tables (git-ignored); `work/scratch` holds the large U-Net maps (~21 GB) |
| `outputs/` | `submission.csv`, `figures/`, `logs/` (git-ignored) |
| `reference/` | SHA-256 of the data and models (`CHECKSUMS.json`), download sources (`DOWNLOADS.json`), the submitted file's SHA-256, reference figures |

## 0. Configuration

Each folder can be moved with an environment variable set before starting Jupyter: `FILAMENT_DATA`, `FILAMENT_MODELS`, `FILAMENT_WORK`,
`FILAMENT_SCRATCH` (~35 GB free; a RAM disk such as `/dev/shm/filament` makes steps 3 to 5 faster), `FILAMENT_OUTPUTS`,
`FILAMENT_DOWNLOADS` (the 7.4 GB archive while it is extracted). Keep the repository
itself on a normal disk: compiled libraries cannot load from a `noexec` RAM disk.

In [ ]:
import os, sys, re, json, time, shutil, hashlib, subprocess
from pathlib import Path

REPO = Path(os.environ.get('FILAMENT_REPO', Path.cwd())).resolve()      # the repository (this notebook's folder)
sys.path.insert(0, str(REPO))
from src import paths                                                  # every folder: data, models, work, scratch, outputs
PY = sys.executable

def _gpus():
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        return list(range(len([l for l in out.splitlines() if l.startswith('GPU ')])))
    except FileNotFoundError:
        return []
GPUS = [int(g) for g in os.environ['GPUS'].split(',')] if os.environ.get('GPUS') else _gpus()
CPUS = int(os.environ.get('CPUS', min(24, os.cpu_count() or 1)))   # worker processes of the CPU steps (results do not depend on it)
RETRAIN = os.environ.get('RETRAIN', '').strip().lower()              # '' = ask (section 0.3); 'yes'/'1' = retrain without asking; 'no' = never
DATA_OK = os.environ.get('DATA_OK', '0') == '1'                      # continue even if the data differ from the checksums
FORCE = os.environ.get('FORCE', '0') == '1'                          # recompute inference outputs even if they exist
FINAL_R = 3                                                          # boundary redraw width of the final submission (px)
LOGS = paths.OUTPUTS / 'logs'; LOGS.mkdir(parents=True, exist_ok=True)
FIG = paths.OUTPUTS / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
for d in (paths.DATA, paths.MODELS, paths.WORK, paths.SCRATCH):
    d.mkdir(parents=True, exist_ok=True)
print(f'repo    {REPO}\ndata    {paths.DATA}\nmodels  {paths.MODELS}\nwork    {paths.WORK}\n'
      f'scratch {paths.SCRATCH}  ({shutil.disk_usage(paths.SCRATCH).free / 1e9:.0f} GB free)\noutputs {paths.OUTPUTS}\n'
      f'python  {PY}\nGPUs    {GPUS}   CPU workers {CPUS}   retrain: {RETRAIN or "ask"}   force inference: {FORCE}')

### 0.1 Download what is missing

Nothing is downloaded when the files are already there. From a fresh clone, this cell fetches:

1. **Competition data** from Kaggle (photos and annotations, ~0.7 GB) into `data/train` and `data/test`. This needs the `kaggle` package,
   your API token and the competition rules accepted.
2. **Hugging Face files** at the exact commits used (`reference/DOWNLOADS.json`): the deformable-attention kernel for inference, and the
   starting weights for training. Every later step runs with `HF_HUB_OFFLINE=1`, so the code's requests for "main" resolve to these commits.
   If Hugging Face rate-limits anonymous downloads, set `HF_TOKEN`.
3. **`filament_models.tar`** (7.4 GB): the trained models (`models/`) and the shipped inference outputs (`work/`). It is taken from
   `downloads/` if present, else from Google Drive (`gdrive` in `DOWNLOADS.json`), else from a private Kaggle dataset (`kaggle_dataset`).
   Its SHA-256 is checked before it is extracted.

Every file is then checked by SHA-256 in section 0.3.

In [ ]:
import tarfile, zipfile
DL = json.loads((REPO / 'reference/DOWNLOADS.json').read_text())
DOWNLOADS = paths.DOWNLOADS; DOWNLOADS.mkdir(parents=True, exist_ok=True)

def kaggle(*args):
    exe = shutil.which('kaggle') or str(Path(PY).with_name('kaggle'))
    r = subprocess.run([exe, *args], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f'kaggle {" ".join(args)} failed (token in ~/.kaggle/kaggle.json? rules accepted?):\n{(r.stdout + r.stderr)[-1500:]}')

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

# 1. competition data
if not paths.TRAIN_IMAGES.is_dir() or not paths.TEST_IMAGES.is_dir():
    print(f"downloading the competition data ({DL['kaggle_competition']}, ~0.7 GB)...")
    tmp = DOWNLOADS / 'kaggle_data'; shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
    kaggle('competitions', 'download', '-c', DL['kaggle_competition'], '-p', str(tmp))
    for zp in tmp.glob('*.zip'):
        with zipfile.ZipFile(zp) as z:
            z.extractall(tmp)
    src = next(tmp.rglob('train_images')).parent.parent                 # the folder holding train/ and test/
    for part in ('train', 'test'):
        (paths.DATA / part).mkdir(parents=True, exist_ok=True)
        for item in (src / part).iterdir():
            if not (paths.DATA / part / item.name).exists():
                shutil.move(str(item), str(paths.DATA / part / item.name))
    shutil.rmtree(tmp)
print('data:', len(list(paths.TRAIN_IMAGES.glob('*.jpeg'))), 'training photos,', len(list(paths.TEST_IMAGES.glob('*.jpeg'))), 'test photos')

# 2. Hugging Face files at pinned commits; refs/main points at them (all later steps run offline)
HF_HUB = REPO / '.cache/huggingface/hub'
for repo, spec in DL['huggingface'].items():
    d = HF_HUB / ('models--' + repo.replace('/', '--'))
    for f in spec['files']:                                             # exact files at the pinned commit (no repository listing)
        if (d / 'snapshots' / spec['revision'] / f).is_file():
            continue
        from huggingface_hub import hf_hub_download
        for attempt in range(6):                                        # anonymous requests can be rate-limited: wait and retry
            try:
                hf_hub_download(repo, f, revision=spec['revision'], cache_dir=str(HF_HUB)); break
            except Exception as e:
                if attempt == 5:
                    raise SystemExit(f'Hugging Face download failed for {repo}/{f}: {e}\nRetry later or set HF_TOKEN.')
                print(f'  {repo}/{f}: {type(e).__name__}, retrying in {30 * (attempt + 1)} s'); time.sleep(30 * (attempt + 1))
    (d / 'refs').mkdir(parents=True, exist_ok=True)
    if not (d / 'refs/main').is_file() or (d / 'refs/main').read_text().strip() != spec['revision']:
        (d / 'refs/main').write_text(spec['revision'])
print('Hugging Face files:', ', '.join(f"{r} @ {s['revision'][:8]}" for r, s in DL['huggingface'].items()))

# 3. the models archive
A = DL['archive']; arc = DOWNLOADS / A['name']
if (paths.MODELS / A['marker']).exists():
    print(f"{A['name']}: already extracted")
else:
    if not arc.is_file() and DL.get('gdrive'):
        import gdown
        print(f"downloading {A['name']} from Google Drive (7.4 GB)...")
        g = DL['gdrive']
        gdown.download(g if g.startswith('http') else None, id=None if g.startswith('http') else g, output=str(arc), quiet=False, fuzzy=True)
    if not arc.is_file() and DL.get('kaggle_dataset'):
        print(f"downloading {A['name']} from the Kaggle dataset {DL['kaggle_dataset']}...")
        kaggle('datasets', 'download', '-d', DL['kaggle_dataset'], '-f', A['name'], '-p', str(DOWNLOADS))
        if (DOWNLOADS / (A['name'] + '.zip')).is_file():
            with zipfile.ZipFile(DOWNLOADS / (A['name'] + '.zip')) as z:
                z.extractall(DOWNLOADS)
            (DOWNLOADS / (A['name'] + '.zip')).unlink()
    if not arc.is_file():
        print(f"{A['name']}: NOT FOUND. Put it in downloads/ or set its Google Drive link in reference/DOWNLOADS.json; "
              'section 0.3 will then list every model as missing.')
    else:
        got = sha256(arc)
        if got != A['sha256']:
            raise SystemExit(f"{A['name']}: SHA-256 {got[:16]}... differs from {A['sha256'][:16]}...: download it again")
        with tarfile.open(arc) as t:                                    # members: models/... and work/...
            for m in t.getmembers():
                top, rest = m.name.split('/', 1)
                m.name = str((paths.MODELS if top == 'models' else paths.WORK) / rest)
                t.extract(m, '/', filter='data')
        print(f"{A['name']}: SHA-256 verified, extracted into {paths.MODELS} and {paths.WORK}")
        if os.environ.get('KEEP_ARCHIVES', '0') != '1':
            arc.unlink()

### 0.2 Environment and helpers

* **Checks.** Python packages, GPUs, data files, and that the attention kernel is not on a `noexec` mount.
* **`BASE_ENV`.** The environment of every step: the folders of `src/paths.py`, `HF_HOME` / `TORCH_HOME` inside the repository, offline Hugging Face.
* **Helpers.** `sh()` runs one command, with its output in `outputs/logs/<name>.log`. `gpu_queue()` spreads independent jobs over the GPUs
  (with `CUDA_VISIBLE_DEVICES`) and skips jobs whose output exists.

In [ ]:
import numpy as np, pandas as pd
import torch, cv2, transformers, lightgbm
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPUs visible:', torch.cuda.device_count(),
      '| transformers', transformers.__version__, '| lightgbm', lightgbm.__version__, '| numpy', np.__version__)
k = REPO / '.cache/huggingface/hub/models--kernels-community--deformable-detr'
if k.exists() and os.statvfs(k.resolve()).f_flag & os.ST_NOEXEC:
    raise SystemExit(f'{k} is on a noexec mount: keep the repository on a normal disk (only FILAMENT_SCRATCH may be a RAM disk)')
for p in (paths.FOLDS_JSON, paths.ANNOTATIONS, paths.TRAIN_IMAGES, paths.TEST_IMAGES):
    assert p.exists(), f'missing {p}'
FOLDS = json.loads(paths.FOLDS_JSON.read_text())['folds']
n_test = len(list(paths.TEST_IMAGES.glob('*.jpeg')))
print('training photos per fold:', [len(f) for f in FOLDS], '| test photos:', n_test)

BASE_ENV = dict(os.environ, PYTHONPATH=os.pathsep.join(p for p in (str(REPO), os.environ.get('PYTHONPATH')) if p),
                PYTHONUNBUFFERED='1', HF_HOME=str(REPO / '.cache/huggingface'),
                TORCH_HOME=str(REPO / '.cache/torch'), HF_HUB_OFFLINE='1', HF_HUB_ENABLE_HF_TRANSFER='0',
                OPENBLAS_NUM_THREADS='1', OPENCV_FOR_THREADS_NUM='1', FILAMENT_CPUS=str(CPUS),
                FILAMENT_DATA=str(paths.DATA), FILAMENT_MODELS=str(paths.MODELS), FILAMENT_WORK=str(paths.WORK),
                FILAMENT_SCRATCH=str(paths.SCRATCH), FILAMENT_OUTPUTS=str(paths.OUTPUTS))
BASE_ENV.pop('CUDA_VISIBLE_DEVICES', None)


def sh(cmd, name, env=None, tail=6):
    """Run one command (list) from the repository, output to outputs/logs/<name>.log; raise if it fails; print the last lines."""
    t0 = time.time()
    with open(LOGS / f'{name}.log', 'a') as fh:
        fh.write(f'\n### {time.strftime("%F %T")} {" ".join(map(str, cmd))}\n'); fh.flush()
        rc = subprocess.run([str(c) for c in cmd], cwd=REPO, env=env or BASE_ENV, stdout=fh, stderr=subprocess.STDOUT).returncode
    lines = [l for l in (LOGS / f'{name}.log').read_text(errors='ignore').splitlines() if l.strip() and 'Warning' not in l]
    print(f'[{name}] exit {rc} in {(time.time() - t0) / 60:.1f} min'); print('\n'.join('    ' + l[:220] for l in lines[-tail:]))
    if rc:
        raise RuntimeError(f'{name} failed (exit {rc}); see {LOGS / (name + ".log")}')


def gpu_queue(jobs, per_gpu=1, poll=15):
    """jobs: dicts(name, cmd, done (Path: the job's output), env). Jobs whose output exists are skipped; the others run with at most
    `per_gpu` jobs per GPU (CUDA_VISIBLE_DEVICES set to the GPU's index). Raises if any job fails."""
    todo = [j for j in jobs if not Path(j['done']).exists()]
    print(f'{len(jobs) - len(todo)} of {len(jobs)} jobs already done; {len(todo)} to run on GPUs {GPUS}')
    if todo and not GPUS:
        raise SystemExit('no GPU found')
    load = {g: 0 for g in GPUS}; running = {}; failed = []; t0 = time.time()
    while todo or running:
        while todo and min(load.values()) < per_gpu:
            g = min(load, key=load.get); j = todo.pop(0); load[g] += 1
            env = dict(j.get('env') or BASE_ENV, CUDA_VISIBLE_DEVICES=str(g))
            fh = open(LOGS / f"{j['name']}.log", 'a')
            p = subprocess.Popen([str(c) for c in j['cmd']], cwd=REPO, env=env, stdout=fh, stderr=subprocess.STDOUT)
            running[p] = (j, g, fh, time.time()); print(f"  start {j['name']} on GPU {g}", flush=True)
        time.sleep(poll)
        for p in [p for p in running if p.poll() is not None]:
            j, g, fh, ts = running.pop(p); fh.close(); load[g] -= 1
            ok = p.returncode == 0 and Path(j['done']).exists()
            print(f"  {'done' if ok else 'FAILED'} {j['name']} ({(time.time() - ts) / 60:.0f} min)", flush=True)
            if not ok:
                failed.append(j['name'])
    print(f'queue finished in {(time.time() - t0) / 60:.0f} min')
    if failed:
        raise RuntimeError(f'failed jobs: {failed} (logs in {LOGS})')

### 0.3 Integrity check: data, models and shipped outputs (SHA-256)

`reference/CHECKSUMS.json` holds the SHA-256 of everything the published result was built on:

* the folds, the annotations and **each of the 707 training photos and 180 test photos**;
* the **40 checkpoints** in `models/`;
* the shipped inference outputs in `work/`.

This cell recomputes every hash (~1 min) and decides what happens next.

* **Data.** If the folds, the annotations or the photos differ, the models do not correspond to these data. The notebook stops and lists the
  differences. Set `DATA_OK=1` to continue anyway.
* **Checkpoints.** One that is present with the right hash is used as is. One that is **missing or different** is listed, with an estimate
  of its training time. Nothing is retrained until you confirm: answer `y`, or set `RETRAIN=1` when running headless. After confirmation:
  * a different file is renamed `*.mismatch-<time>` (kept, never deleted);
  * the inference outputs built from that model are deleted, so they are recomputed with the new model;
  * section 1 retrains exactly these models.
* **Shipped inference outputs.** A file with a different hash is deleted and recomputed in section 2.
* **Models retrained here.** Their SHA-256 is recorded in `models/retrained.json` after training; later runs accept them (status
  "retrained here") and do not compare the outputs rebuilt from them with the reference checksums.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
CHECK = json.loads((REPO / 'reference/CHECKSUMS.json').read_text())

def hash_all(files):
    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(lambda p: sha256(p) if Path(p).is_file() else None, files))

t0 = time.time()
# --- data (paths relative to data/)
groups = {'folds.json': ['folds.json'], 'annotations': ['train/MAGFiLO_1.0_Annotations_kaggle2026_train.json'],
          'training photos': [n for n in CHECK['data'] if n.startswith('train/train_images/')],
          'test photos': [n for n in CHECK['data'] if n.startswith('test/test_images/')]}
names = [n for g in groups.values() for n in g]
got = dict(zip(names, hash_all([paths.DATA / n for n in names])))
rows, bad_data = [], []
for g, ns in groups.items():
    miss = [n for n in ns if got[n] is None]; diff = [n for n in ns if got[n] is not None and got[n] != CHECK['data'][n]]
    rows.append((g, len(ns), len(ns) - len(miss) - len(diff), len(miss), len(diff)))
    if miss or diff:
        bad_data += [(g, n, 'missing' if n in miss else 'different') for n in miss + diff]
print(pd.DataFrame(rows, columns=['data', 'files', 'identical', 'missing', 'different']).to_string(index=False))
if bad_data:
    print('DIFFERENT DATA:', bad_data[:10], '...' if len(bad_data) > 10 else '')
    if not DATA_OK:
        raise SystemExit('The data differ from those the models were built on (list above). Fix them, or set DATA_OK=1 to continue anyway.')

# --- checkpoints (paths relative to models/)
TRAIN_HOURS = {'mask2former': 5.5, 'mask2former_limb': 5.5, 'maskrcnn': 1.7, 'unet_cnx': 1.3, 'unet_cnxL': 2.7, 'unet_r34': 1.2,
               'roi_detector': 3.1, 'roi_refiner': 1.9}                 # one 12 GB GPU
family = lambda rel: rel.split('/')[0].rsplit('-f', 1)[0]             # noqa: E731
ck = list(CHECK['checkpoints'])
RETRAINED = paths.MODELS / 'retrained.json'                          # SHA-256 of the models retrained here (accepted by later runs)
local = json.loads(RETRAINED.read_text()) if RETRAINED.is_file() else {}
hashes = dict(zip(ck, hash_all([paths.MODELS / r for r in ck])))
status = {r: 'ok' if hashes[r] == CHECK['checkpoints'][r] else 'retrained here' if hashes[r] is not None and hashes[r] == local.get(r)
          else 'missing' if hashes[r] is None else 'hash differs' for r in ck}
print(f"\ncheckpoints: {sum(v == 'ok' for v in status.values())} of {len(ck)} present with the expected SHA-256, "
      f"{sum(v == 'retrained here' for v in status.values())} retrained here "
      f"({sum((paths.MODELS / r).stat().st_size for r in ck if (paths.MODELS / r).is_file()) / 1e9:.1f} GB hashed in {time.time() - t0:.0f} s)")
TO_TRAIN = [r for r in ck if status[r] not in ('ok', 'retrained here')]


def derived(rel):
    """The inference outputs built from one checkpoint."""
    name = rel.split('/')[0]; fam, k = name.rsplit('-f', 1)
    W, S = paths.WORK, paths.SCRATCH
    if fam in ('mask2former', 'mask2former_limb'):
        out = [W / f'views/{fam}/fold{k}_{s}.json' for s in ('val', 'test')] + [W / f'pools/fold{k}_{s}.json' for s in ('val', 'test')]
        if fam == 'mask2former':
            out += [W / f'mask2former/fold{k}_{x}.json' for x in ('test_d4', 'val_d4', 'test_single')] + [paths.MODELS / name / 'e050_cache.json']
        return out
    if fam == 'maskrcnn':
        return [W / f'banks/maskrcnn-f{k}']
    if fam.startswith('unet_'):
        a = fam[5:]
        return [S / f'maps/{u}/test{a}.npy{x}' for u in (f'train_f{k}', 'test') for x in ('', '.json')]
    return [W / f'banks/roi-f{k}']                                     # roi_detector / roi_refiner


# --- shipped inference outputs: a file with a different hash is deleted (section 2 recomputes it); a missing one is recomputed
cache = [(paths.WORK / r, h) for r, h in CHECK['work'].items()] + [(paths.MODELS / r, h) for r, h in CHECK['model_outputs'].items()]
local_outputs = [str(d) for r in ck if status[r] == 'retrained here' for d in derived(r)]   # rebuilt from a retrained model: not compared
cache = [(p, h) for p, h in cache if not any(str(p) == d or str(p).startswith(d + '/') for d in local_outputs)]
ch = hash_all([p for p, _ in cache])
bad_cache = [p for (p, h), g in zip(cache, ch) if g is not None and g != h]
for p in bad_cache:
    p.unlink()
print(f"inference outputs: {sum(g == h for (_, h), g in zip(cache, ch))} of {len(cache)} identical, "
      f"{sum(g is None for g in ch)} absent (recomputed in section 2), {len(bad_cache)} different (deleted, recomputed)")

if not TO_TRAIN:
    TRAIN_MISSING = False
    print('every checkpoint verified: nothing will be retrained')
else:
    tab = pd.DataFrame([(r, status[r], TRAIN_HOURS[family(r)]) for r in TO_TRAIN], columns=['checkpoint', 'status', 'training h (1 GPU)'])
    print(tab.to_string(index=False))
    hours = tab['training h (1 GPU)'].sum()
    wall = max(tab['training h (1 GPU)'].max(), hours / max(len(GPUS), 1))      # one model cannot be split across GPUs
    print(f'-> {len(TO_TRAIN)} models to retrain: ~{hours:.1f} GPU-hours (~{wall:.1f} h on {len(GPUS)} GPUs). '
          f'Their inference outputs will be recomputed too.')
    ans = RETRAIN
    if ans not in ('y', 'yes', '1', 'n', 'no', '0'):
        try:
            ans = input('Retrain these models now? [y/N] ').strip().lower()
        except (EOFError, OSError):                                     # headless: no prompt possible
            ans = ''
    if ans not in ('y', 'yes', '1'):
        raise SystemExit('Stopped, nothing retrained. Restore the listed checkpoints (delete them and rerun section 0.1 to extract '
                         'the archive again), or confirm: answer y, or set RETRAIN=1 (headless).')
    stamp = time.strftime('%Y%m%d-%H%M%S')
    for r in TO_TRAIN:
        p = paths.MODELS / r
        if p.exists() or p.is_symlink():
            p.rename(p.with_name(p.name + f'.mismatch-{stamp}'))              # kept, never deleted
        (p.parent / 'done.json').unlink(missing_ok=True)                  # the evidence-model trainer skips a model with done.json
        for d in derived(r):
            if d.is_dir() and not d.is_symlink():
                shutil.rmtree(d)
            elif d.exists() or d.is_symlink():
                d.unlink()
    TRAIN_MISSING = True
    print(f'confirmed: {len(TO_TRAIN)} checkpoints set aside, their inference outputs deleted; section 1 retrains them')

## 1. Training (only models whose checkpoint is missing or changed, after confirmation)

Every model family is trained 5 times. Model *k* is trained on the photos of the other 4 folds, so its predictions on fold *k* are out of fold.
With the downloaded checkpoints, every cell of this section only lists what exists and moves on.

| Family | Checkpoint (fold *k*) | Recipe |
|---|---|---|
| 1a `mask2former` | `models/mask2former-f{k}/model_e050.pt` | Swin-S Mask2Former, COCO instance start, grayscale input, 50 epochs, batch 8 |
| 1b `mask2former_limb` | `models/mask2former_limb-f{k}/model_e050.pt` | same, 3-channel limb-corrected input and crop zoom 0.8-1.25 |
| 1c `maskrcnn`, `unet_cnx` / `unet_cnxL` / `unet_r34` | `models/maskrcnn-f{k}/model.pt`, `models/unet_*-f{k}/ema.pt` | Mask R-CNN R50-FPN v2; U-Nets with ConvNeXt-T / ResNet34 encoders |
| 1d `roi_detector`, `roi_refiner` | `models/roi_{detector,refiner}-f{k}/weights.pt` | Faster R-CNN v2 on tiles + ResNet18 U-Net refiner on box crops |

### 1a. Mask2Former, grayscale input: 5 folds

* **Model.** Swin-S Mask2Former (`transformers`), started from the COCO instance-segmentation weights, on 2048 px solar photos at native resolution.
* **Training.** 50 epochs, batch 8 crops (gradient accumulation, micro-batch 1), bf16, transformers' own Mask2Former loss, evaluated on the
  held-out fold every 5 epochs. The deformable attention runs on a prebuilt CUDA kernel and the mask loss samples matched masks in chunks;
  both are checked for equality with the reference code before use.
* **Memory.** On a 12 GB GPU the backbone is checkpointed on whole frames. After an out-of-memory crash the launcher retries with
  checkpointing everywhere (same maths, slower), and it resumes from `last.pt` if the process dies.
* **Epoch.** Out of fold, PQ rises up to the last epoch (table below), so **epoch 50** is used.

In [ ]:
def train_mask2former(family, workers):
    jobs = [dict(name=f'train_{family}-f{k}', cmd=[PY, '-m', 'src.training.mask2former', family, str(k), '--workers', str(workers)],
                 done=paths.model_dir(family, k) / 'model_e050.pt') for k in range(5)]
    missing = [j['name'] for j in jobs if not Path(j['done']).exists()]
    print(f'{family}: checkpoints present for folds {[k for k in range(5) if (paths.model_dir(family, k) / "model_e050.pt").is_file()]}')
    if missing and not TRAIN_MISSING:
        raise SystemExit(f'missing {missing}: not confirmed in section 0.3')
    gpu_queue(jobs, per_gpu=1)             # one Mask2Former training per 12 GB GPU


def epoch_table(family):
    """Out-of-fold PQ of each fold model every 5 epochs (single view): from the training's own sweep files when the model was trained
    here, else the reference table of the published models."""
    rows = {}
    for k in range(5):
        for f in sorted(paths.model_dir(family, k).glob('e0*_sweep.json')):
            rows.setdefault(int(f.stem[1:4]), {})[f'fold {k}'] = json.loads(f.read_text())['best']['pq']
    if not rows:
        ref = pd.read_csv(REPO / 'reference/mask2former_oof_by_epoch.csv')
        ref = ref[ref.family == family]
        rows = {e: {f'fold {k}': v for k, v in zip(g.fold, g.oof_pq_single_view)} for e, g in ref.groupby('epoch')}
    t = pd.DataFrame(rows).T.sort_index(); t['mean'] = t.mean(axis=1)
    return t.round(4)

train_mask2former('mask2former', workers=12)
epoch_table('mask2former')

### 1b. Mask2Former, limb-corrected input: 5 folds

Same recipe, with two changes in the input pipeline:

* **3-channel input** (`FILAMENT_PREP=limb3`): the raw photo; the solar disk divided by its radial median profile (limb-darkening
  correction, robust to filaments), found by a Hough circle on the limb; and a CLAHE copy of that flattened disk.
* **Crop zoom 0.8-1.25** (`FILAMENT_SCALE_JITTER=1.25`): random scale jitter of the training crops.

The recipe fixes epoch 50, as for the first family; epochs 45 and 50 are within 0.001 out of fold. Alone this family is not better
(out of fold, single view, epoch 50: 0.4606 vs 0.4630). Fused with the first family it adds ~25% candidates to each photo's pool.

In [ ]:
train_mask2former('mask2former_limb', workers=6)
epoch_table('mask2former_limb')

### 1c. Evidence models: Mask R-CNN + 3 U-Nets, 20 models

They are not used as predictors. They give the scorer **evidence**: does another model see the same filament?

* **U-Nets** (`src/training/train_unet.py`). Two output channels, filament probability and spine (centre line). Batch 8, 512 px crops,
  BCE + 0.5 Dice + 0.3 spine BCE, EMA weights. `unet_cnx`: ConvNeXt-T, 24k steps. `unet_cnxL`: ConvNeXt-T, 48k steps. `unet_r34`: ResNet34, 30k steps.
* **Mask R-CNN** (`src/training/train_maskrcnn.py`). 12k iterations, batch 4, 1024 px crops, SGD lr 0.01, COCO start.
* `src/training/evidence_models.py` builds a training cache once (~8 GB in scratch). It gives each fold a metadata file that holds out
  exactly that fold, then queues the 20 trainings (2 U-Nets or 1 Mask R-CNN per 12 GB GPU). A model with `done.json` is skipped.

In [ ]:
evidence = [f'{m}-f{k}' for m in ('maskrcnn', 'unet_cnx', 'unet_cnxL', 'unet_r34') for k in range(5)]
missing = [m for m in evidence if not (paths.MODELS / m / 'done.json').is_file()]
print(f'{len(evidence) - len(missing)} of 20 Mask R-CNN / U-Net models present; missing: {missing or "none"}')
if missing:
    assert TRAIN_MISSING, 'models missing: not confirmed in section 0.3'
    sh([PY, '-m', 'src.training.evidence_models', 'train', '--gpus', ','.join(map(str, GPUS))], 'train_evidence_models')

### 1d. ROI detector + refiner: 10 models

* **Detector.** Faster R-CNN v2 (torchvision, COCO start) on 9 tiles per photo.
* **Refiner.** A ResNet18 U-Net (ImageNet start) that redraws the mask inside each detected box's context crop.
* **Training.** Settings in `configs/roi/training.json`. A checkpoint every 1000 steps; a resume draws the same samples.
* **Workspace.** The trainer prepares `work/roi/`: links to its configs, manifests, photos and annotations, plus its caches. The
  rasterised targets are cached once (`--prepare`) before the jobs start; up to 4 of these jobs share a 12 GB GPU.

In [ ]:
jobs = [dict(name=f'train_roi_{s}-f{k}', cmd=[PY, '-m', 'src.training.train_roi', '--stage', s, '--fold', str(k)],
             done=paths.MODELS / f'roi_{s}-f{k}' / 'weights.pt',
             env=dict(BASE_ENV, OMP_NUM_THREADS='2', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True'))
        for s in ('detector', 'refiner') for k in range(5)]
if not all(Path(j['done']).exists() for j in jobs):
    assert TRAIN_MISSING, 'ROI weights missing: not confirmed in section 0.3'
    sh([PY, '-m', 'src.training.train_roi', '--prepare'], 'roi_targets')   # target cache built once, before the parallel jobs
gpu_queue(jobs, per_gpu=4)

### Checkpoints used by the pipeline (40)

In [ ]:
CKPT = pd.DataFrame([(r, (paths.MODELS / r).is_file(), round((paths.MODELS / r).stat().st_size / 1e6) if (paths.MODELS / r).is_file() else None)
                     for r in CHECK['checkpoints']], columns=['checkpoint', 'present', 'MB'])
assert CKPT.present.all(), CKPT[~CKPT.present]
if TRAIN_MISSING:                                                  # record the retrained models' SHA-256 for later runs
    local = json.loads(RETRAINED.read_text()) if RETRAINED.is_file() else {}
    local.update({r: sha256(paths.MODELS / r) for r in TO_TRAIN})
    RETRAINED.write_text(json.dumps(local, indent=1))
    print(f'{len(TO_TRAIN)} retrained checkpoints recorded in {RETRAINED}')
print(f'{len(CKPT)} checkpoints, {CKPT.MB.sum() / 1000:.1f} GB')
CKPT

## 2. Inference: candidates and evidence (out of fold on the 707 training photos, and on the 180 test photos)

Each output below is skipped when it exists (`FORCE=1` deletes and recomputes it). The downloaded archive contains every output except the
U-Net maps.

| Output | Where | In the archive |
|---|---|---|
| 2a Mask2Former single-view and 8-view caches | `models/mask2former-f{k}/e050_cache.json`, `work/mask2former/` | yes |
| 2b raw 8-view candidates of both families | `work/views/{mask2former,mask2former_limb}/fold{k}_{val,test}.json` | yes |
| 2c fused 16-view pools | `work/pools/fold{k}_{val,test}.json` | yes |
| 2d Mask R-CNN banks, U-Net maps | `work/banks/maskrcnn-f{k}/`, `work/scratch/maps/` | banks yes, maps **no** (~21 GB, ~25 min) |
| 2e ROI banks | `work/banks/roi-f{k}/` | yes |

In [ ]:
def forced(files):
    """FORCE=1: delete the given outputs so they are recomputed."""
    if FORCE:
        for p in map(Path, files):
            if p.is_dir() and not p.is_symlink():
                shutil.rmtree(p)
            elif p.exists() or p.is_symlink():
                p.unlink()

M2F_ENV = dict(BASE_ENV, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', NUMEXPR_NUM_THREADS='1')

### 2a. Mask2Former single-view and 8-view candidate caches (`src/inference/m2f_caches.py`)

* **Held-out photos, single view** (`e050_cache.json`). Written by the training run itself at epoch 50 (bf16). If it is missing, it is rebuilt
  with `--split val{k} --bf16`, which is equivalent but not guaranteed byte-identical.
* **Test, single view** (fp32) and **8 views** (D4: 4 rotations x mirror, mapped back and fused; held-out photos in bf16, test in fp32).
* **Use.** The scorer reads the single-view pool as a second opinion on each candidate. The 8-view pools of this family only serve as a size
  comparison in 2c.

In [ ]:
jobs = []
for k in range(5):
    ck = paths.model_dir('mask2former', k) / 'model_e050.pt'
    for kind, args in (('test_d4', ['--split', 'test', '--d4']), ('val_d4', ['--split', f'val{k}', '--d4', '--bf16']), ('test_single', ['--split', 'test'])):
        out = paths.WORK / f'mask2former/fold{k}_{kind}.json'
        jobs.append(dict(name=f'cache_mask2former-f{k}_{kind}', done=out, env=M2F_ENV,
                         cmd=[PY, '-m', 'src.inference.m2f_caches', '--checkpoint', ck, *args, '--workers', '1', '--threads', '16', '--out', out]))
    out = paths.model_dir('mask2former', k) / 'e050_cache.json'
    jobs.append(dict(name=f'cache_mask2former-f{k}_val_single', done=out, env=M2F_ENV,
                     cmd=[PY, '-m', 'src.inference.m2f_caches', '--checkpoint', ck, '--split', f'val{k}', '--bf16', '--workers', '1', '--threads', '16', '--out', out]))
forced([j['done'] for j in jobs if 'val_single' not in j['name']])     # the training's own held-out cache is never deleted
gpu_queue(jobs, per_gpu=1)          # fp32 inference peaks at ~7.6 GB: one per 12 GB GPU

### 2b. Raw 8-view candidates of both families (`src/inference/m2f_views.py`)

Same prediction as the caches (score floor 0.02, bf16), but the candidates of the 8 views are kept **per view**, not fused, so the two
families can be fused together in 2c. Every fold model runs on its own held-out photos (`val`) and on the 180 test photos (`test`):
20 jobs, ~17 min per family on 4 GPUs.

In [ ]:
jobs = [dict(name=f'views_{fam}-f{k}_{split}', cmd=[PY, '-m', 'src.inference.m2f_views', fam, str(k), split],
             done=paths.WORK / f'views/{fam}/fold{k}_{split}.json',
             env=dict(BASE_ENV, OMP_NUM_THREADS='2', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True'))
        for fam in ('mask2former', 'mask2former_limb') for k in range(5) for split in ('val', 'test')]
forced([j['done'] for j in jobs])
gpu_queue(jobs, per_gpu=1)

### 2c. Two-family candidate pools (`src/inference/pools.py`, CPU, ~1 min)

For each fold *k* and each photo, the 16 views (8 per family) are fused:

* candidates are linked when their mask IoU exceeds 0.5;
* a group's mask is the pixel vote (pixels in at least half of its members);
* a group's score is the mean member score x (0.5 + 0.5 x members / 16), so a candidate seen in many views ranks higher.

In [ ]:
pool_files = [paths.WORK / f'pools/fold{k}_{s}.json' for k in range(5) for s in ('val', 'test')]
forced(pool_files)
if not all(p.is_file() for p in pool_files):
    sh([PY, '-m', 'src.inference.pools'], 'pools')
pool = {s: sum(len(v) for k in range(5) for v in json.loads((paths.WORK / f'pools/fold{k}_{s}.json').read_text()).values()) for s in ('val', 'test')}
print(f"fused pools: {pool['val']} candidates on the 707 out-of-fold photos, {pool['test']} on test (5 fold models x 180 photos)")

### 2d. Mask R-CNN candidate banks and U-Net probability maps (`src/training/evidence_models.py infer`)

* **Mask R-CNN** (`maskrcnn-f{k}`). Full 2048 px, fp32: candidate masks and scores on fold *k*'s photos and on the test photos.
* **U-Nets.** Both channels (filament probability, spine), D4 mean, bf16, stored as uint8 arrays `[photos, 2, 2048, 2048]` in
  `work/scratch/maps`. A test map is the rounded mean of the 5 fold models' maps.
* **Size.** ~21 GB, ~25 min on 4 GPUs. To recompute, delete `maps/` entirely: a test map that exists stops the fold maps of the same U-Net
  from being recomputed.

In [ ]:
MAPS = paths.SCRATCH / 'maps'
bank_done = [paths.WORK / f'banks/maskrcnn-f{k}/{s}/manifest.json' for k in range(5) for s in ('train', 'test')]
map_done = [MAPS / f'train_f{k}/test{a}.npy.json' for k in range(5) for a in ('cnx', 'cnxL', 'r34')] + [MAPS / f'test/test{a}.npy.json' for a in ('cnx', 'cnxL', 'r34')]
if FORCE:
    forced([paths.WORK / f'banks/maskrcnn-f{k}' for k in range(5)] + [MAPS])
if not all(p.is_file() for p in bank_done + map_done):
    sh([PY, '-m', 'src.training.evidence_models', 'infer', '--gpus', ','.join(map(str, GPUS))], 'infer_evidence_models')
print('Mask R-CNN banks:', sum(p.is_file() for p in bank_done), 'of 10 | U-Net maps:', sum(p.is_file() for p in map_done), 'of 18',
      f'| {sum(f.stat().st_size for f in MAPS.rglob("*.npy")) / 1e9:.1f} GB')

### 2e. ROI candidate banks (`src/inference/roi_banks.py`)

The detector on 9 tiles, global NMS, then the refiner on each box's context crop; each refined mask is a candidate with 24 features.
Pair *k* runs on fold *k*'s photos and on the test photos.

In [ ]:
jobs = [dict(name=f'banks_roi-f{k}', cmd=[PY, '-m', 'src.inference.roi_banks', str(k)], done=paths.WORK / f'banks/roi-f{k}/test/manifest.json',
             env=dict(BASE_ENV, OMP_NUM_THREADS='4')) for k in range(5)]
if FORCE:
    forced([paths.WORK / f'banks/roi-f{k}' for k in range(5)])
gpu_queue(jobs, per_gpu=1)

## 3. The learned candidate scorer (`src/scorer/selector.py`, CPU)

The pools hold a correct candidate for most annotated filaments. The hard part is choosing which candidates to keep, and a LightGBM model
learns that choice from the out-of-fold candidates.

**Target.** For one candidate, the mean over the photo's annotator records of its IoU with the filament it overlaps most, counted only
above 0.5 (the metric's matching threshold), weighted by the photo's number of records. This is the question PQ asks of each record.

**76 numbers per candidate.**

| Block | # | What |
|---|---|---|
| base | 30 | Fused 16-view score, agreement with the single-view pool (best IoU, its score, number of matches), area, box, elongation, distance from the disk centre, intensity and ring contrast, rank, competitors (overlapping candidates, their scores, best IoU) |
| `maskrcnn` | 5 | Agreement with the Mask R-CNN candidates (best IoU, its score, best score above IoU 0.5, count above 0.3, IoU x score). Training photos: only the model that did not see the photo; test: mean over the 5 |
| `spine_cnx`, `spine_cnxL`, `spine_r34` | 3 x 12 | What each U-Net's spine map says inside and around the mask: mean, std, 10th percentile, share above 0.5 and 0.25, ring mean and max, and how the mask covers the map's own blobs at 0.5 and 0.25 |
| `roi` | 5 | Agreement with the ROI detector + refiner candidates (same 5 numbers as `maskrcnn`) |

**Model.** LightGBM regression: 300 rounds, 15 leaves, learning rate 0.05, at least 60 rows per leaf, feature / bagging fraction 0.8,
L2 1.0, averaged over 5 seeds. It is **cross-fitted**: fold *k*'s candidates are scored by a model trained on the other 4 folds.

In [ ]:
SCORER = paths.WORK / 'scorer'; OUTD = SCORER / 'final'
shutil.rmtree(SCORER, ignore_errors=True)                     # always rebuilt from the current candidates and evidence
sh([PY, '-m', 'src.scorer.selector', 'features'], 'scorer_features')          # 30 base numbers (+ the target on training photos)
sh([PY, '-m', 'src.scorer.selector', 'evidence'], 'scorer_evidence')          # Mask R-CNN and U-Net spine blocks
sh([PY, '-m', 'src.scorer.selector', 'roi_evidence'], 'scorer_roi_evidence')  # ROI detector + refiner block
z = np.load(SCORER / 'oof_rows.npz', allow_pickle=True)
X = np.concatenate([z['X']] + [np.load(SCORER / f'extra_{b}_oof.npy') for b in ('maskrcnn', 'spine_cnx', 'spine_cnxL', 'spine_r34', 'roi')], 1)
print(f'{X.shape[0]} out-of-fold candidates on {len(z["names"])} photos, {X.shape[1]} numbers each')

## 4. Selection on out-of-fold photos, then on test

**Selection.** Among a photo's candidates, keep the set that maximises the sum of (q - tau), with no two kept masks overlapping by more than
theta of the smaller one. The solution is exact (scipy MILP per connected component, `src/scorer/select.py`). Kept masks are painted in
descending q (a pixel goes to the best mask), and masks under 100 px are dropped.

**Choice of tau and theta** (out of fold). A grid over the 707 photos picks **tau 0.225, theta 0.1**. A nested version (tau chosen on
4 folds, applied to the 5th) checks the choice is not overfitted.

**Test.** The scorer is retrained on all out-of-fold rows (5 seeds) and scores each fold model's test candidates. The 5 fold models'
candidates are fused (IoU link 0.5, pixel vote 0.5). A group's q is the mean over the 5 models of that model's best member (0 if none), so
a candidate proposed by few models is marked down. Selection uses the out-of-fold tau and theta; nothing is tuned on test.

In [ ]:
sh([PY, '-m', 'src.scorer.selector', 'fit'], 'scorer_fit', tail=4)            # cross-fitted q, tau / theta grid, nested check
rep = json.loads((OUTD / 'oof_report.json').read_text())
print(f"chosen on out-of-fold: tau {rep['best']['tau']}, theta {rep['best']['theta']} -> PQ as emitted {rep['best']['pq']:.4f} "
      f"(nested {rep['nested_pq']:.4f})")
print('PQ by tau:', {k: round(v, 4) for k, v in rep['grid'].items()})
sh([PY, '-m', 'src.scorer.selector', 'test'], 'scorer_test', tail=6)           # one selection per fold model
sh([PY, '-m', 'src.scorer.selector', 'ensemble'], 'scorer_ensemble', tail=4)   # 5 fold models fused -> submission_selector_ens5.csv

## 5. Full-resolution boundaries (`src/scorer/redraw_oof.py` out of fold, `src/scorer/redraw.py` on test)

Mask2Former predicts masks at 1/4 resolution; the U-Nets predict probability maps at full resolution. Each kept mask **keeps its extent**,
but its boundary is redrawn within *r* px from P, the mean of the cnxL and r34 filament-probability maps:

    M' = erode(M, r)  +  (P > t) inside dilate(M, r),   t = the threshold at which this best reproduces M (grid 0.10-0.90)

The submission uses r = 3 px (out of fold +0.0021 over the masks as emitted; 1 px is equal within noise). On test, the maps are the mean of
the 5 fold U-Nets.

In [ ]:
sh([PY, '-m', 'src.scorer.redraw_oof'], 'redraw_oof', tail=4, env=dict(BASE_ENV, FILAMENT_REDRAW_RADII=str(FINAL_R)))
FINAL = OUTD / f'submission_selector_ens5_fullres{FINAL_R}.csv'
sh([PY, '-m', 'src.scorer.redraw', OUTD / 'submission_selector_ens5.csv', FINAL, str(FINAL_R)], 'redraw_test', tail=2)

## 6. Results and checks

* **Out-of-fold PQ** on the 707 training photos, pooled exactly like the competition metric (each annotator's record matched
  separately, TP / FP / FN pooled over all records, PQ divided once).
* **Reproduction check.** SHA-256 of `outputs/submission.csv` against the submitted file (`reference/SHA256SUMS`).

In [ ]:
from src.evaluation.oof import pq
files = {'as emitted': (OUTD / 'submission_selector_ens5.csv', 'oof_counts_asemitted.npy'),
         f'{FINAL_R} px redraw': (FINAL, f'oof_counts_fullres{FINAL_R}.npy')}
files_csv = {label: f for label, (f, _) in files.items()}
rows = []
for label, (csv_path, counts_file) in files.items():
    t = np.load(OUTD / counts_file).sum(0)
    rows.append(dict(file=label, oof_pq=round(pq(t), 4), tp=int(t[0]), fp=int(t[1]), fn=int(t[2])))
res = pd.DataFrame(rows).set_index('file')
for label, (f, _) in files.items():
    res.loc[label, 'test masks'] = len(pd.read_csv(f))
ref = dict(l.split()[::-1] for l in (REPO / 'reference/SHA256SUMS').read_text().splitlines())
res.loc[f'{FINAL_R} px redraw', 'identical to submitted'] = sha256(FINAL) == ref.get('submission.csv')
shutil.copy(FINAL, paths.OUTPUTS / 'submission.csv')
sub = pd.read_csv(paths.OUTPUTS / 'submission.csv')
assert list(sub.columns) == ['filament_id', 'segmentation_rle'] and sub.filament_id.is_unique and sub.segmentation_rle.notna().all()
print(f'outputs/submission.csv: {len(sub)} filaments on {sub.filament_id.str.split("_").str[0].nunique()} of {n_test} test photos '
      f'(sha256 {sha256(FINAL)[:16]}...)')
res

## 7. Illustrations

1. **How each stage moved the out-of-fold PQ.** Earlier stages are recorded results of the same protocol; the last row is computed here.
2. **Out-of-fold predictions on training photos.** The final pipeline's filaments (filled blue) on three training photos, each predicted
   by models and a scorer that never saw it, with one annotator's filaments (orange outlines). The photos are the best, the median and
   the worst by out-of-fold match rate.
3. **The boundary redraw, close up.** One test mask before (white) and after (blue) the 3 px redraw from the U-Net maps.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from pycocotools import mask as mu
INK, MUTED, GRID, BLUE, ORANGE, SURFACE = '#0b0b0b', '#898781', '#e1e0d9', '#2a78d6', '#eb6834', '#fcfcfb'
plt.rcParams.update({'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'axes.edgecolor': '#c3c2b7', 'text.color': INK,
                     'axes.labelcolor': '#52514e', 'xtick.color': MUTED, 'ytick.color': INK, 'font.size': 10})

# --- 1. out-of-fold PQ at each stage (dot plot: the axis is zoomed, so no bars)
stages = [('One Mask2Former family, fixed rule, single view', 0.4628),
          ('+ 8 views + learned candidate scorer', 0.4734),
          ('+ boundary redraw from the U-Net maps', 0.4748),
          ('+ all evidence models trained on the folds', 0.4768),
          ('+ second Mask2Former family (this notebook)', float(res.loc[f'{FINAL_R} px redraw', 'oof_pq']))]
fig, ax = plt.subplots(figsize=(9.5, 3.4))
ys = np.arange(len(stages))[::-1]; vals = [s[1] for s in stages]
ax.hlines(ys, min(vals) - 0.004, vals, color=GRID, lw=2)
ax.scatter(vals[:-1], ys[:-1], s=64, color=MUTED, zorder=3); ax.scatter(vals[-1:], ys[-1:], s=80, color=BLUE, zorder=3)
for y, v in zip(ys, vals):
    ax.text(v + 0.0005, y, f'{v:.4f}', va='center', color=INK)
ax.set_yticks(ys, [s[0] for s in stages]); ax.set_xlim(min(vals) - 0.004, max(vals) + 0.004)
ax.grid(axis='x', color=GRID, lw=0.8); ax.set_axisbelow(True); ax.spines[['top', 'right', 'left']].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.set_title('Out-of-fold PQ (707 training photos) at each stage', loc='left', fontsize=12)
fig.tight_layout(); fig.savefig(FIG / 'score_progression.png', dpi=130); plt.show()

In [ ]:
# --- 2. out-of-fold predictions of the final pipeline vs one annotator on three training photos (best / median / worst)
from src.scorer.oof_jobs import load_jobs, base_masks
from src.scorer.redraw import redraw
from src.scorer.paint import paint
from src.evaluation.oof import native
names, folds_of, oof_jobs = load_jobs()
C = np.load(OUTD / f'oof_counts_fullres{FINAL_R}.npy')                       # per photo, pooled over its records: tp, fp, fn, IoU sum
per = pd.DataFrame({'i': range(len(names)), 'tp': C[:, 0], 'fp': C[:, 1], 'fn': C[:, 2]})
per['match_rate'] = per.tp / (per.tp + 0.5 * per.fp + 0.5 * per.fn).clip(lower=1e-9)
per = per[(per.tp + per.fn) >= 4].sort_values('match_rate')
picks = [per.iloc[-1], per.iloc[len(per) // 2], per.iloc[0]]
fig, axes = plt.subplots(1, 3, figsize=(16, 6.4))
for ax, row, label in zip(axes, picks, ('best', 'median', 'worst')):
    i = int(row.i); name = names[i]; k = int(folds_of[i])
    records, entries, q, ovl, tau, theta = oof_jobs[i]
    _, masks = base_masks(entries, q, ovl, tau, theta)
    files = json.loads((paths.SCRATCH / f'maps/train_f{k}/testcnxL.npy.json').read_text())['files']
    U = np.mean([np.asarray(np.load(paths.SCRATCH / f'maps/train_f{k}/test{a}.npy', mmap_mode='r')[files.index(name), 0]).astype(np.float32)
                 for a in ('cnxL', 'r34')], 0)
    final = paint([redraw(m, U, FINAL_R) for m in masks])
    pred = mu.decode(final) if final else np.zeros((2048, 2048, 0), np.uint8)
    rec = [native(r) for r in records[0]]                                   # the photo's first annotator record
    truth = mu.decode(rec) if rec else np.zeros((2048, 2048, 0), np.uint8)
    hits = int((np.asarray(mu.iou(final, rec, [0] * len(rec))) > 0.5).sum()) if final and rec else 0
    img = cv2.imread(str(paths.TRAIN_IMAGES / name), cv2.IMREAD_GRAYSCALE)
    ys_, xs_ = np.nonzero(pred.any(2) | truth.any(2))                       # crop around every filament, square, at least 640 px
    cy, cx = (ys_.min() + ys_.max()) // 2, (xs_.min() + xs_.max()) // 2
    h = int(max(640, ys_.max() - ys_.min() + 160, xs_.max() - xs_.min() + 160)) // 2
    y0, x0 = int(np.clip(cy - h, 0, 2048 - 2 * h)), int(np.clip(cx - h, 0, 2048 - 2 * h))
    sl = (slice(y0, y0 + 2 * h), slice(x0, x0 + 2 * h))
    over = np.dstack([img[sl]] * 3).astype(np.float32) / 255
    m = pred.any(2)[sl]
    over[m] = 0.45 * over[m] + 0.55 * np.array([0x2a, 0x78, 0xd6]) / 255
    edge = cv2.morphologyEx(truth.any(2)[sl].astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)).astype(bool)
    over[edge] = np.array([0xeb, 0x68, 0x34]) / 255
    ax.imshow(over); ax.axis('off')
    ax.set_title(f'{label} of {len(per)} training photos: {Path(name).stem} (fold {k})\nours {pred.shape[2]}, annotator {truth.shape[2]}: '
                 f'matched {hits}, extra {pred.shape[2] - hits}, missed {truth.shape[2] - hits}', loc='left', fontsize=10)
handles = [matplotlib.patches.Patch(color=BLUE, alpha=0.7, label='our filaments (out of fold)'),
           matplotlib.patches.Patch(facecolor='none', edgecolor=ORANGE, lw=2, label='one annotator')]
fig.legend(handles=handles, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Out-of-fold predictions on training photos vs one annotator (IoU > 0.5 = matched)', x=0.01, ha='left', fontsize=12)
fig.tight_layout(rect=(0, 0.05, 1, 0.95)); fig.savefig(FIG / 'oof_predictions.png', dpi=110); plt.show()

In [ ]:
# --- 3. boundary redraw close up: the largest mask of a test photo, before and after
before = pd.read_csv(files_csv['as emitted']); after = sub
for d in (before, after):
    d['photo'] = d.filament_id.str.split('_', n=1).str[0]
photo = after.photo.value_counts().index[0]
b = mu.decode([{'size': [2048, 2048], 'counts': s} for s in before.segmentation_rle[before.photo == photo]])
a = mu.decode([{'size': [2048, 2048], 'counts': s} for s in after.segmentation_rle[after.photo == photo]])
i = int(b.reshape(-1, b.shape[2]).sum(0).argmax()); j = int(np.argmax([(a[..., t] & b[..., i]).sum() for t in range(a.shape[2])]))
ys, xs = np.nonzero(b[..., i] | a[..., j]); c = 40
y0, y1, x0, x1 = max(ys.min() - c, 0), min(ys.max() + c, 2048), max(xs.min() - c, 0), min(xs.max() + c, 2048)
img = cv2.imread(str(paths.TEST_IMAGES / f'{photo}.jpeg'), cv2.IMREAD_GRAYSCALE)[y0:y1, x0:x1]
fig, ax = plt.subplots(figsize=(7, 7 * (y1 - y0) / max(x1 - x0, 1) + 0.6))
ax.imshow(img, cmap='gray')
ax.contour(b[y0:y1, x0:x1, i], levels=[0.5], colors='white', linewidths=1.6)
ax.contour(a[y0:y1, x0:x1, j], levels=[0.5], colors=BLUE, linewidths=1.6)
ax.legend(handles=[matplotlib.lines.Line2D([], [], color='white', lw=2, label='Mask2Former mask (1/4 resolution, upsampled)'),
                   matplotlib.lines.Line2D([], [], color=BLUE, lw=2, label=f'after the {FINAL_R} px redraw from the U-Net maps')],
          loc='lower left', framealpha=0.85)
ax.set_title(f'{photo}: boundary redraw on one filament ({int(b[..., i].sum())} -> {int(a[..., j].sum())} px)', loc='left', fontsize=11)
ax.axis('off'); fig.tight_layout(); fig.savefig(FIG / 'redraw_closeup.png', dpi=130); plt.show()

## 8. Submitting and what was tried

**Submit** (Kaggle CLI, token in `~/.kaggle`): `kaggle competitions submit -c filament-segmentation-2026 -f outputs/submission.csv -m "two-model ensemble, 3 px"`

**Tried and rejected on out-of-fold PQ:** gap filling, averaged scorers, multi-scale inference, mask width changes, the ROI refiner applied to
our masks, merging split filaments, size-dependent thresholds, limb filtering, more scorer rounds or leaves, contrast features, an HRNet
evidence block, clDice training, model-agreement features, a vote of fold-wise scorers, and a pixel-level CNN scorer (ResNet18 on candidate
crops, stacked into LightGBM: out of fold -0.0010).